# Dry Bean Classification — Model Training

In [2]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef, confusion_matrix)

## 1. Load data

In [4]:
try:
    from ucimlrepo import fetch_ucirepo
    beans = fetch_ucirepo(id=602)
    X = beans.data.features
    y = beans.data.targets['Class']
except Exception as e:
    print("Download failed, loading local Excel file:", e)
    df = pd.read_excel('Dry_Bean_Dataset.xlsx')   # needs: pip install openpyxl
    X = df.drop(columns=['Class'])
    y = df['Class']

print(X.shape)
print("Missing values:", X.isnull().sum().sum())
print(y.value_counts())

(13611, 16)
Missing values: 0
Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64


## 2. Train/test split, save test data, scale features

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=27)

test_df = X_test.copy()
test_df['Class'] = y_test.values
test_df.to_csv('../test_data.csv', index=False)   # saved in project root

enc = LabelEncoder().fit(y_train)
y_tr, y_te = enc.transform(y_train), enc.transform(y_test)

scaler = StandardScaler().fit(X_train)
X_tr, X_te = scaler.transform(X_train), scaler.transform(X_test)
print(X_tr.shape, X_te.shape)

(10888, 16) (2723, 16)


## 3. Evaluation function (6 metrics)

In [8]:
def evaluate(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)
    return {
        'Accuracy':  accuracy_score(y, pred),
        'AUC':       roc_auc_score(y, proba, multi_class='ovr'),
        'Precision': precision_score(y, pred, average='weighted'),
        'Recall':    recall_score(y, pred, average='weighted'),
        'F1':        f1_score(y, pred, average='weighted'),
        'MCC':       matthews_corrcoef(y, pred),
    }

## 4. Train, evaluate and save all 5 models

In [10]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, random_state=27),
    'kNN':                 KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes':         GaussianNB(),
    'SVM':                 SVC(kernel='rbf', probability=True, random_state=27),
}
results = {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    results[name] = evaluate(m, X_te, y_te)
    joblib.dump(m, name.lower().replace(' ', '_') + '.pkl')
    print("done:", name)

joblib.dump(scaler, 'scaler.pkl')
joblib.dump(enc, 'label_encoder.pkl')

table = pd.DataFrame(results).T.round(4)
table

done: Logistic Regression
done: Decision Tree
done: kNN
done: Naive Bayes


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


done: SVM


,Accuracy,AUC,Precision,Recall,F1,MCC
Logistic Regression,0.9188,0.9941,0.9190,0.9188,0.9189,0.9019
Decision Tree,0.9053,0.9640,0.9059,0.9053,0.9054,0.8854
kNN,0.9148,0.9876,0.9151,0.9148,0.9149,0.8969
Naive Bayes,0.8909,0.9901,0.8923,0.8909,0.8910,0.8686
SVM,0.9269,0.9947,0.9270,0.9269,0.9269,0.9116


## 5. Comparison table in Markdown (for README)

In [12]:
print(table.to_markdown())   # needs: pip install tabulate

|                     |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:--------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression |     0.9188 | 0.9941 |      0.919  |   0.9188 | 0.9189 | 0.9019 |
| Decision Tree       |     0.9053 | 0.964  |      0.9059 |   0.9053 | 0.9054 | 0.8854 |
| kNN                 |     0.9148 | 0.9876 |      0.9151 |   0.9148 | 0.9149 | 0.8969 |
| Naive Bayes         |     0.8909 | 0.9901 |      0.8923 |   0.8909 | 0.891  | 0.8686 |
| SVM                 |     0.9269 | 0.9947 |      0.927  |   0.9269 | 0.9269 | 0.9116 |


## 6. Confusion matrix of the best model

In [14]:
best_name = table['MCC'].idxmax()
print("Best model by MCC:", best_name)
pd.DataFrame(confusion_matrix(y_te, models[best_name].predict(X_te)),
             index=enc.classes_, columns=enc.classes_)

Best model by MCC: SVM


,BARBUNYA,BOMBAY,CALI,DERMASON,HOROZ,SEKER,SIRA
BARBUNYA,242,0,14,0,1,3,5
BOMBAY,0,104,0,0,0,0,0
CALI,10,0,302,0,9,1,4
DERMASON,0,0,0,661,0,12,36
HOROZ,0,0,5,5,370,0,6
SEKER,2,0,0,7,0,383,14
SIRA,2,0,0,45,13,5,462
